# Jeu de devinette : Père Fouras vs Laurent Jalabert

Dans ce notebook, nous allons simuler le duel légendaire entre le Père Fouras et Laurent Jalabert en utilisant Semantic Kernel avec des agents conversationnels.

In [1]:
# Bloc 1 - Installation et imports
%pip install semantic-kernel python-dotenv --quiet
import os
import logging
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.agents.strategies import KernelFunctionTerminationStrategy
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import KernelArguments

# Configuration des logs
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger('FortBoyard')

# Chargement des variables d'environnement
env_path = os.path.join(os.path.dirname(os.path.abspath(".")), ".env")
if os.path.exists(env_path):
    load_dotenv(env_path)
else:
    load_dotenv()



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Configuration des agents

In [2]:
# Bloc 2 - Création du kernel
MOT_A_DEVINER = "anticonstitutionnellement"

def create_kernel():
    kernel = Kernel()
    model_id = os.getenv("OPENAI_CHAT_MODEL_ID", "gpt-5-mini")
    kernel.add_service(OpenAIChatCompletion(
        service_id="openai",
        ai_model_id=model_id,
        api_key=os.getenv("OPENAI_API_KEY")
    ))
    return kernel


## Définition des prompts

In [3]:
# Bloc 3 - Prompts des agents
PERE_FOURAS_PROMPT = f"""
Tu es le Père Fouras de Fort Boyard. 
Tu dois faire deviner le mot '{MOT_A_DEVINER}'. 
Utilise des charades et réponses énigmatiques. 
Ne révèle jamais directement le mot !
"""

LAURENT_JALABERT_PROMPT = """
Tu es Laurent Jalabert. 
Tu dois deviner le mot en posant des questions fermées (Oui/Non).
Sois perspicace et stratégique dans tes questions.
"""

## Création des agents avec stratégies personnalisées

In [4]:
# Bloc 4 - Définition des agents
pere_fouras = ChatCompletionAgent(
    kernel=create_kernel(),
    name="Pere_Fouras",
    instructions=PERE_FOURAS_PROMPT,
)

laurent_jalabert = ChatCompletionAgent(
    kernel=create_kernel(),
    name="Laurent_Jalabert",
    instructions=LAURENT_JALABERT_PROMPT,
)

## Stratégie de terminaison personnalisée

In [5]:
from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy
from semantic_kernel.contents.chat_message_content import ChatMessageContent

# Bloc 5 - Logique de terminaison
class FortBoyardTerminationStrategy(TerminationStrategy):
    """Arrête la partie si le mot est deviné"""
    
    async def should_agent_terminate(
        self, 
        agent: ChatCompletionAgent, 
        history: list[ChatMessageContent], 
        cancellation_token = None
    ) -> bool:
        if not history:
            return False
        
        last_message = str(history[-1].content).lower()
        return MOT_A_DEVINER in last_message

## Configuration du groupe de discussion

In [6]:
# Bloc 6 - Configuration corrigée
chat = AgentGroupChat(
    agents=[pere_fouras, laurent_jalabert],
    termination_strategy=FortBoyardTerminationStrategy(
        agents=[laurent_jalabert],  # Définit explicitement les agents
        maximum_iterations=20       # Définit le nombre max d'itérations
    )
)


## Lancement de la partie !

In [7]:
from semantic_kernel.contents import AuthorRole, ChatMessageContent

# Bloc 7 - Exécution du jeu
async def jouer_partie():
    logger.info("Depart du duel Pere Fouras vs Laurent Jalabert !")
    logger.info(f"Mot a deviner : {MOT_A_DEVINER.upper()}")
    
    # Message initial pour lancer la conversation
    await chat.add_chat_message(
        ChatMessageContent(role=AuthorRole.USER, content="Pere Fouras, donne-moi un indice !")
    )
    
    async for message in chat.invoke():
        role = message.role
        logger.info(f"[{role}] : {message.content}")
    
    logger.info("Partie terminee !")

await jouer_partie()

2026-05-10 01:26:58,737 [INFO] Depart du duel Pere Fouras vs Laurent Jalabert !


2026-05-10 01:26:58,739 [INFO] Mot a deviner : ANTICONSTITUTIONNELLEMENT


2026-05-10 01:26:58,740 [INFO] Adding `1` agent chat messages


2026-05-10 01:26:58,742 [INFO] Selected agent at index 0 (ID: b12596d6-1228-4c9d-92cd-386ac5d57fe9, name: Pere_Fouras)


2026-05-10 01:26:58,743 [INFO] Invoking agent Pere_Fouras


...

2026-05-10 01:27:15,720 [INFO] OpenAI usage: CompletionUsage(completion_tokens=30, prompt_tokens=1377, total_tokens=1407, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


2026-05-10 01:27:15,721 [INFO] Evaluating termination criteria for d92fb529-8bf1-4e5f-bac5-839f6541ce0c


2026-05-10 01:27:15,722 [INFO] Evaluated criteria for d92fb529-8bf1-4e5f-bac5-839f6541ce0c, should terminate: True


2026-05-10 01:27:15,722 [INFO] [AuthorRole.ASSISTANT] : **Est-ce que le mot que tu as en tête, c’est “anticonstitutionnellement” ? (Oui/Non)**


2026-05-10 01:27:15,723 [INFO] Partie terminee !
